In [1]:
import h5py
import numpy as np
import pandas as pd
import os
from datetime import datetime

In [2]:
with h5py.File('E:\\Thesis\\ICESat\\Tests\\ATL08_20200701061628_00900805_005_01.h5', 'r') as hf:
        print('List of arrays in this file: \n', hf.keys())

List of arrays in this file: 
 <KeysViewHDF5 ['METADATA', 'ancillary_data', 'ds_geosegments', 'ds_metrics', 'ds_surf_type', 'gt1l', 'gt1r', 'gt2l', 'gt2r', 'gt3l', 'gt3r', 'orbit_info', 'quality_assessment']>


In [18]:
IceSat = h5py.File(f, 'r')
gt1l= IceSat['gt1l']
LS = gt1l['land_segments']
Canopy = LS['canopy']

In [163]:
Latitude = np.array(list(LS['latitude'])).flatten()
Longitude = np.array(list(LS['longitude'])).flatten()
df['Latitude'] = Latitude
df['Longitude'] = Longitude

In [43]:
directory = 'E:\\Thesis\\ICESat\\ICESat2019'
test = os.listdir(directory)

testname = test[650]

x = testname.split('_')

datestring = x[1]
print(datestring)

date = datetime.strptime(datestring, '%Y%m%d%H%M%S')
print(date)
                         
            



20190825083829
2019-08-25 08:38:29


In [ ]:
directory = 'E:\\Thesis\\ICESat\\ICESatV2'
ExportDF = pd.DataFrame()
filecount = len([entry for entry in os.listdir(directory) if os.path.isfile(os.path.join(directory, entry))]) / 2

counter = 0

for filename in os.listdir(directory):
    f = os.path.join(directory, filename)
    if f.endswith('.h5'):
        IceSat = h5py.File(f, 'r')
        counter += 1
        #Flags and coordinates
        df = pd.DataFrame()
        

        for beam in list(IceSat.keys()):

            #Only include strong beams
            if beam == 'gt1l' or beam == 'gt2l' or beam == 'gt3l':
                beams = IceSat[beam]
                try:
                    LS = beams['land_segments']

                except:
                    pass # doing nothing on exception
                tempdf = pd.DataFrame()
                Keylist = LS.keys()

                #Set latitude and longitude before looping (important to do this here to set the size of tempdf)
                tempdf['latitude'] = np.array(list(LS['latitude'])).flatten()
                tempdf['longitude'] = np.array(list(LS['longitude'])).flatten()

                #Loop through each beam and get quality flags and canopy cahracteristics, and add to temp DF
                for key in Keylist:
                    if key == 'brightness_flag':
                        tempdf[str(key)] = np.array(LS[key])

                    elif key == 'layer_flag':
                        tempdf[str(key)] = np.array(LS[key])
                    elif key == 'night_flag':
                        tempdf[str(key)] = np.array(LS[key])
                        
                    elif key == 'asr':
                        tempdf[str(key)] = np.array(LS[key])
                        
                    elif key == 'canopy':
                        Canopy = LS[key]
                        CanopyKeys = LS[key].keys()
                        for key in CanopyKeys:        
                            if key == 'canopy_h_metrics':
                                percentile = 10
                                try:
                                    arraylist = np.hsplit(Canopy[key], np.size(Canopy[key][1]))
                                except:
                                    pass
                                for array in arraylist:
                                    name = str(key) + str(percentile)
                                    try:
                                        tempdf[str(key) + 'P' + str(percentile)] = np.array(array)
                                    except:
                                        pass
                                    percentile+=5

                            elif key == 'h_canopy_uncertainty':
                                tempdf[str(key)] = np.array(Canopy[key])

                            elif key == 'canopy_openness':
                                tempdf[str(key)] = np.array(Canopy[key])

                            elif key == 'segment_cover':
                                tempdf[str(key)] = np.array(Canopy[key])

                            elif key == 'toc_roughness':
                                tempdf[str(key)] = np.array(Canopy[key])

                #Add the beam
                tempdf['beam'] = str(beam)
                
                #Add the date (extracted from the filename)
                filename_split = filename.split('_')
                datestring = filename_split[1]
                tempdf['date'] = datetime.strptime(datestring, '%Y%m%d%H%M%S')

                #Concat tempDF to the export DF
                df = pd.concat([df,tempdf])
                ExportDF = pd.concat([ExportDF,df])
                
                print("Processed " + str(counter) + "/" + str(filecount), end='\r')

ExportDF
        

In [46]:
#Filter out unfavourable shots
filtered = ExportDF.loc[(ExportDF['night_flag'] == 1) & (ExportDF['brightness_flag'] == 0)]

In [47]:
filtered.size

740990280

In [48]:
filtered.to_csv("E:\\Thesis\\ICESat\\CSVs\\ICESat_ForKoreen2.csv")